# sphinxcontrib_serializinghtml

sphinxcontrib-serializinghtml 2.0.0 is Sphinx's `json` and `pickle` builders: each page's rendered HTML body, title
and navigation written as data, for a site generator (Django, Flask) to render. These cells read the JSON pages, the
global context, the pickle output, and the registration.

In [ ]:
# Example 1: the json builder writes one .fjson per page with the rendered body.
import json
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'Demo'\n")
(src / "index.rst").write_text("Demo\n====\n\n.. toctree::\n\n   page\n\nIntro with *emphasis*.\n")
(src / "page.rst").write_text("Page\n====\n\nA page about :index:`widgets`.\n")
rc = build_main(["-q", "-W", "-b", 'json', str(src), str(src / "_out")])
out = src / "_out"
print("rc", rc, sorted(p.name for p in out.iterdir()))
assert rc == 0
page = json.loads((out / 'index.fjson').read_text())
print(sorted(page))
assert page['title'] == 'Demo' and 'Intro with <em>emphasis</em>' in page['body']

In [ ]:
# Example 2: the global context carries the project-wide values a site generator needs.
import json
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'Demo'\nrelease = '1.2'\n")
(src / "index.rst").write_text("Demo\n====\n\n.. toctree::\n\n   page\n\nIntro with *emphasis*.\n")
(src / "page.rst").write_text("Page\n====\n\nA page about :index:`widgets`.\n")
rc = build_main(["-q", "-W", "-b", 'json', str(src), str(src / "_out")])
out = src / "_out"
print("rc", rc, sorted(p.name for p in out.iterdir()))
assert rc == 0
ctx = json.loads((out / 'globalcontext.json').read_text())
print({k: ctx[k] for k in ('project', 'release')})
assert ctx['project'] == 'Demo' and ctx['release'] == '1.2'

In [ ]:
# Example 3: page navigation is data too: the next/previous links between the two pages.
import json
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'Demo'\n")
(src / "index.rst").write_text("Demo\n====\n\n.. toctree::\n\n   page\n\nIntro with *emphasis*.\n")
(src / "page.rst").write_text("Page\n====\n\nA page about :index:`widgets`.\n")
rc = build_main(["-q", "-W", "-b", 'json', str(src), str(src / "_out")])
out = src / "_out"
print("rc", rc, sorted(p.name for p in out.iterdir()))
assert rc == 0
page = json.loads((out / 'page.fjson').read_text())
print(page.get('prev'))
assert page['prev']['link'] == '../' and 'Page' == page['title']

In [ ]:
import pickle
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'Demo'\n")
(src / "index.rst").write_text("Demo\n====\n\n.. toctree::\n\n   page\n\nIntro with *emphasis*.\n")
(src / "page.rst").write_text("Page\n====\n\nA page about :index:`widgets`.\n")
rc = build_main(["-q", "-W", "-b", 'pickle', str(src), str(src / "_out")])
out = src / "_out"
print("rc", rc, sorted(p.name for p in out.iterdir()))
assert rc == 0
page = pickle.loads((out / 'index.fpickle').read_bytes())
assert page['title'] == 'Demo' and 'emphasis' in page['body']

In [ ]:
# Example 5: both builders are registered by one extension, with their own file suffixes.
import sphinxcontrib.serializinghtml as serializinghtml
from sphinxcontrib.serializinghtml import JSONHTMLBuilder, PickleHTMLBuilder

print(serializinghtml.__version__, JSONHTMLBuilder.out_suffix, PickleHTMLBuilder.out_suffix)
assert serializinghtml.__version__ == "2.0.0"
assert (JSONHTMLBuilder.name, PickleHTMLBuilder.name) == ("json", "pickle")
assert (JSONHTMLBuilder.out_suffix, PickleHTMLBuilder.out_suffix) == (".fjson", ".fpickle")